# 03 - Exploratory Data Analysis: Samples & Responses

This notebook provides comprehensive analysis of the VLM Router evaluation dataset.

**Sections:**
1. Dataset Overview & Table Statistics
2. Sample Distribution Analysis
3. Model Performance Comparison
4. Latency & Cost Analysis
5. Confidence Score Analysis
6. GPU Metrics Analysis
7. Response Quality Analysis
8. Cross-Model Correlation
9. Image Display Examples

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
from PIL import Image
import io

from ares.db.connection import get_engine
from ares.configs.db_config import TABLES, MODEL_NAMES

engine = get_engine()
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Color palette for models
MODEL_COLORS = {
    'deepseek_ocr': '#FF6B6B',
    'qwen2_5_vl_3b': '#4ECDC4',
    'qwen2_5_vl_7b': '#45B7D1',
    'qwen3_vl_8b_thinking': '#96CEB4',
    'gemma_3_27b': '#FFEAA7'
}

print('Connected!')

---
## 1. Dataset Overview & Table Statistics

**Purpose:** Get a high-level view of the database contents - row counts, column counts, and data freshness.

In [ ]:
# Table row counts
table_stats = []
with engine.connect() as conn:
    for name, table in TABLES.items():
        count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
        table_stats.append({'Table': name, 'Table Name': table, 'Rows': count})

df_tables = pd.DataFrame(table_stats)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_tables['Table'], df_tables['Rows'], color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'])
ax.set_xlabel('Number of Rows')
ax.set_title('Database Table Sizes', fontsize=14, fontweight='bold')
for bar, val in zip(bars, df_tables['Rows']):
    ax.text(val + max(df_tables['Rows'])*0.01, bar.get_y() + bar.get_height()/2, 
            f'{val:,}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

df_tables

---
## 2. Sample Distribution Analysis

**Purpose:** Understand the composition of our dataset - which Cauldron configs, task types, and data splits are represented.

In [ ]:
# Load samples data
df_samples = pd.read_sql('SELECT * FROM vlm_samples', engine)
print(f'Total samples: {len(df_samples):,}')

In [ ]:
# Distribution by source config
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of source configs
config_counts = df_samples['source_config'].value_counts()
config_counts.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Number of Samples')
axes[0].set_title('Samples by Cauldron Config', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()

# Pie chart of data splits
split_counts = df_samples['data_split'].value_counts()
colors = ['#3498db', '#2ecc71', '#e74c3c']
axes[1].pie(split_counts, labels=split_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=[0.02]*len(split_counts))
axes[1].set_title('Data Split Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nSplit counts:')
print(split_counts)

In [ ]:
# Distribution by router_task and ground_truth_type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Task distribution
task_counts = df_samples['router_task'].value_counts()
task_counts.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Samples by Router Task', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Ground truth type
gt_counts = df_samples['ground_truth_type'].value_counts()
gt_counts.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='black')
axes[1].set_ylabel('Count')
axes[1].set_title('Samples by Ground Truth Type', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Prompt length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Characters
df_samples['txt_prompt_length_chars'].hist(bins=50, ax=axes[0], color='teal', edgecolor='black', alpha=0.7)
axes[0].axvline(df_samples['txt_prompt_length_chars'].median(), color='red', linestyle='--', label=f'Median: {df_samples["txt_prompt_length_chars"].median():.0f}')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Prompt Length Distribution (Characters)', fontsize=13, fontweight='bold')
axes[0].legend()

# Words
df_samples['txt_prompt_length_words'].hist(bins=50, ax=axes[1], color='darkorange', edgecolor='black', alpha=0.7)
axes[1].axvline(df_samples['txt_prompt_length_words'].median(), color='red', linestyle='--', label=f'Median: {df_samples["txt_prompt_length_words"].median():.0f}')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Prompt Length Distribution (Words)', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Model Performance Comparison

**Purpose:** Compare accuracy, F1 scores, and overall performance across the 5 VLM models.

In [ ]:
# Load responses
query = '''SELECT r.*, s.source_config, s.router_task, s.ground_truth_type, s.data_split
           FROM vlm_responses r
           JOIN vlm_samples s ON r.sample_id = s.sample_id'''
df_resp = pd.read_sql(query, engine)
print(f'Total responses: {len(df_resp):,}')

In [ ]:
# Model performance summary
model_perf = df_resp.groupby('model_name').agg({
    'is_correct': ['sum', 'mean'],
    'score_f1': 'mean',
    'score_exact_match': 'mean',
    'confidence_score': 'mean',
    'latency_ms': 'mean',
    'estimated_cost_usd': 'sum',
    'sample_id': 'count'
}).round(4)
model_perf.columns = ['Correct', 'Accuracy', 'F1 Mean', 'Exact Match', 'Confidence', 'Latency (ms)', 'Total Cost', 'Samples']
model_perf = model_perf.sort_values('Accuracy', ascending=False)
model_perf

In [ ]:
# Accuracy comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy
acc = df_resp.groupby('model_name')['is_correct'].mean().sort_values(ascending=False)
colors = [MODEL_COLORS.get(m, 'gray') for m in acc.index]
acc.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(acc):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# F1 Score
f1 = df_resp.groupby('model_name')['score_f1'].mean().sort_values(ascending=False)
colors = [MODEL_COLORS.get(m, 'gray') for m in f1.index]
f1.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Mean F1 Score', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=45)

# Mean Confidence
conf = df_resp.groupby('model_name')['confidence_score'].mean().sort_values(ascending=False)
colors = [MODEL_COLORS.get(m, 'gray') for m in conf.index]
conf.plot(kind='bar', ax=axes[2], color=colors, edgecolor='black')
axes[2].set_ylabel('Confidence')
axes[2].set_title('Mean Confidence Score', fontsize=13, fontweight='bold')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy by task and model (heatmap)
pivot_acc = df_resp.pivot_table(
    values='is_correct', 
    index='router_task', 
    columns='model_name', 
    aggfunc='mean'
).round(3) * 100

plt.figure(figsize=(14, 8))
sns.heatmap(pivot_acc, annot=True, fmt='.1f', cmap='RdYlGn', 
            vmin=0, vmax=100, linewidths=0.5, cbar_kws={'label': 'Accuracy %'})
plt.title('Model Accuracy by Task Type (%)', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Task')
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy by ground truth type
pivot_gt = df_resp.pivot_table(
    values='is_correct', 
    index='ground_truth_type', 
    columns='model_name', 
    aggfunc='mean'
).round(3) * 100

plt.figure(figsize=(12, 5))
pivot_gt.plot(kind='bar', figsize=(12, 5), edgecolor='black', width=0.8)
plt.ylabel('Accuracy %')
plt.title('Model Accuracy by Ground Truth Type', fontsize=14, fontweight='bold')
plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 4. Latency & Cost Analysis

**Purpose:** Analyze inference speed and cost efficiency across models.

In [ ]:
# Latency distribution by model
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot of latency
df_resp.boxplot(column='latency_ms', by='model_name', ax=axes[0])
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency Distribution by Model', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
plt.suptitle('')  # Remove default title

# Mean latency bar chart
lat = df_resp.groupby('model_name')['latency_ms'].mean().sort_values()
colors = [MODEL_COLORS.get(m, 'gray') for m in lat.index]
lat.plot(kind='barh', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_xlabel('Mean Latency (ms)')
axes[1].set_title('Mean Latency by Model', fontsize=13, fontweight='bold')
for i, (idx, v) in enumerate(lat.items()):
    axes[1].text(v + lat.max()*0.01, i, f'{v:.0f}ms', va='center')

plt.tight_layout()
plt.show()

print('Latency Statistics (ms):')
print(df_resp.groupby('model_name')['latency_ms'].describe().round(0))

In [ ]:
# Cost analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total cost by model
cost_total = df_resp.groupby('model_name')['estimated_cost_usd'].sum().sort_values(ascending=False)
colors = [MODEL_COLORS.get(m, 'gray') for m in cost_total.index]
cost_total.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_ylabel('Total Cost (USD)')
axes[0].set_title('Total Estimated Cost by Model', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Cost per sample
cost_per = df_resp.groupby('model_name')['estimated_cost_usd'].mean().sort_values(ascending=False)
colors = [MODEL_COLORS.get(m, 'gray') for m in cost_per.index]
cost_per.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_ylabel('Cost per Sample (USD)')
axes[1].set_title('Average Cost per Sample', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Token usage analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, col in enumerate(['input_tokens', 'output_tokens', 'total_tokens']):
    means = df_resp.groupby('model_name')[col].mean().sort_values(ascending=False)
    colors = [MODEL_COLORS.get(m, 'gray') for m in means.index]
    means.plot(kind='bar', ax=axes[idx], color=colors, edgecolor='black')
    axes[idx].set_ylabel('Tokens')
    axes[idx].set_title(f'Mean {col.replace("_", " ").title()}', fontsize=13, fontweight='bold')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 5. Confidence Score Analysis

**Purpose:** Understand model confidence calibration - do confident predictions correlate with correctness?

In [ ]:
# Confidence distribution by model
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, model in enumerate(df_resp['model_name'].unique()[:5]):
    model_data = df_resp[df_resp['model_name'] == model]['confidence_score'].dropna()
    axes[idx].hist(model_data, bins=30, edgecolor='black', alpha=0.7, 
                   color=MODEL_COLORS.get(model, 'gray'))
    axes[idx].axvline(model_data.mean(), color='red', linestyle='--', 
                      label=f'Mean: {model_data.mean():.2f}')
    axes[idx].set_xlabel('Confidence Score')
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{model}', fontsize=12, fontweight='bold')
    axes[idx].legend()
    axes[idx].set_xlim(0, 1)

if len(axes) > 5:
    axes[5].axis('off')
    
plt.suptitle('Confidence Score Distributions by Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Confidence vs Accuracy (Calibration)
df_resp['conf_bin'] = pd.cut(df_resp['confidence_score'], bins=10, labels=False)

calib = df_resp.groupby(['model_name', 'conf_bin']).agg({
    'is_correct': 'mean',
    'sample_id': 'count'
}).reset_index()
calib.columns = ['model_name', 'conf_bin', 'accuracy', 'count']

plt.figure(figsize=(12, 6))
for model in calib['model_name'].unique():
    model_data = calib[calib['model_name'] == model]
    plt.plot(model_data['conf_bin']/10 + 0.05, model_data['accuracy'], 
             marker='o', label=model, color=MODEL_COLORS.get(model, 'gray'), linewidth=2)

# Perfect calibration line
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Calibration')
plt.xlabel('Confidence')
plt.ylabel('Actual Accuracy')
plt.title('Confidence Calibration Plot', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 6. GPU Metrics Analysis

**Purpose:** Analyze GPU utilization during inference to understand resource efficiency.

In [ ]:
# GPU metrics summary
gpu_cols = ['gpu_util_percent', 'gpu_mem_used_mb', 'gpu_temp_celsius', 'gpu_power_watts']
gpu_stats = df_resp.groupby('model_name')[gpu_cols].mean().round(2)
gpu_stats

In [ ]:
# GPU metrics visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('gpu_util_percent', 'GPU Utilization (%)', axes[0, 0]),
    ('gpu_mem_used_mb', 'GPU Memory Used (MB)', axes[0, 1]),
    ('gpu_temp_celsius', 'GPU Temperature (°C)', axes[1, 0]),
    ('gpu_power_watts', 'GPU Power (W)', axes[1, 1])
]

for col, title, ax in metrics:
    means = df_resp.groupby('model_name')[col].mean().sort_values(ascending=False)
    colors = [MODEL_COLORS.get(m, 'gray') for m in means.index]
    means.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
    ax.set_ylabel(title)
    ax.set_title(f'Mean {title}', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 7. Response Quality Analysis

**Purpose:** Analyze response characteristics - length, error rates, and refusals.

In [ ]:
# Response success/failure rates
success_rate = df_resp.groupby('model_name').agg({
    'ok': 'mean',
    'is_refusal': 'mean'
}).round(4)
success_rate.columns = ['Success Rate', 'Refusal Rate']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Success rate
colors = [MODEL_COLORS.get(m, 'gray') for m in success_rate.index]
(success_rate['Success Rate'] * 100).plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_ylabel('Success Rate (%)')
axes[0].set_title('Response Success Rate by Model', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 105)
axes[0].tick_params(axis='x', rotation=45)

# Response length distribution
df_resp.boxplot(column='response_length_chars', by='model_name', ax=axes[1])
axes[1].set_ylabel('Response Length (chars)')
axes[1].set_title('Response Length Distribution', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Error analysis
errors = df_resp[df_resp['ok'] == False].groupby('model_name').agg({
    'sample_id': 'count',
    'error_message': lambda x: x.value_counts().head(1).index[0] if len(x) > 0 and x.notna().any() else 'N/A'
})
errors.columns = ['Error Count', 'Most Common Error']
if len(errors) > 0:
    print('Error Summary by Model:')
    display(errors)
else:
    print('No errors found in responses!')

---
## 8. Cross-Model Correlation

**Purpose:** Understand how model performance correlates - do they make similar mistakes?

In [ ]:
# Pivot to get is_correct for each sample x model
pivot_correct = df_resp.pivot(index='sample_id', columns='model_name', values='is_correct')

# Correlation matrix
corr = pivot_correct.astype(float).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Model Agreement Correlation (is_correct)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Model agreement analysis
model_agreement = pivot_correct.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
model_agreement.value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Number of Models Correct')
plt.ylabel('Number of Samples')
plt.title('Model Agreement Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('\nModel Agreement Stats:')
print(f'All 5 models correct: {(model_agreement == 5).sum()} samples ({(model_agreement == 5).mean():.1%})')
print(f'At least 1 correct: {(model_agreement >= 1).sum()} samples ({(model_agreement >= 1).mean():.1%})')
print(f'All wrong: {(model_agreement == 0).sum()} samples ({(model_agreement == 0).mean():.1%})')

---
## 9. Image Display Examples

**Purpose:** Visualize sample images from the dataset.

In [ ]:
def load_image_from_db(image_id):
    query = text('SELECT image_bytes FROM vlm_images WHERE image_id = :id')
    with engine.connect() as conn:
        result = conn.execute(query, {'id': image_id})
        row = result.fetchone()
        if row and row[0]:
            return Image.open(io.BytesIO(row[0]))
    return None

# Display samples from different configs
configs = df_samples['source_config'].unique()[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, cfg in enumerate(configs):
    sample = df_samples[df_samples['source_config'] == cfg].iloc[0]
    img = load_image_from_db(sample['image_id'])
    if img:
        axes[idx].imshow(img)
        axes[idx].set_title(f"{cfg}\nGT: {sample['ground_truth'][:40]}...", fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Sample Images from Different Configs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary Statistics

In [ ]:
print('='*60)
print('DATASET SUMMARY')
print('='*60)
print(f'Total samples: {len(df_samples):,}')
print(f'Total responses: {len(df_resp):,}')
print(f'Unique configs: {df_samples["source_config"].nunique()}')
print(f'Unique tasks: {df_samples["router_task"].nunique()}')
print(f'Models evaluated: {df_resp["model_name"].nunique()}')
print()
print('Overall Accuracy by Model:')
for model in df_resp.groupby('model_name')['is_correct'].mean().sort_values(ascending=False).items():
    print(f'  {model[0]}: {model[1]:.1%}')
print()
print(f'Total estimated cost: ${df_resp["estimated_cost_usd"].sum():.4f}')
print(f'Mean latency: {df_resp["latency_ms"].mean():.0f}ms')